<a href="https://colab.research.google.com/github/L9G9N0/Grandmaster-State-Evaluator/blob/main/Grandmaster_State_Evaluator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chess

import chess.pgn
import io

pgn_data = """
[Result "1-0"]
1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 1-0

[Result "0-1"]
1. e4 c5 2. Nf3 d6 3. d4 cxd4 0-1

[Result "0-1"]
1. e4 c5 2. Nf3 Nc6 3. Bb5 g6 0-1

[Result "1-0"]
1. d4 Nf6 2. c4 e6 3. Nf3 d5 1-0
"""

pgn_io=io.StringIO(pgn_data)
games_list=[]

while True:
  game=chess.pgn.read_game(pgn_io)
  if game is None:
    break

  result =game.headers["Result"]

  board=game.board()
  moves=[]
  for count,move in enumerate(game.mainline_moves()):
    if(count>= 6):
      break;
    moves.append(board.san(move))
    board.push(move)
  opening_sequence=" ".join(moves)

  games_list.append({"Opening": opening_sequence,"Result":result})

import pandas as pd

df_chess=pd.DataFrame(games_list)
print(df_chess)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 24.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=68317477f8b8beba430b7afaa415ad8c5c59007fed735b41607abddda908eedc
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
                Opening Result
0  e4 e5 Nf3 Nc6 Bb5 a6    1-0
1  e4 c5 Nf3 d6 d4 cxd4    0-1
2  e4 c5 Nf3 Nc6 Bb5 g6    0-1
3   d4 Nf6 c4 e6 Nf3 d5    1-0


In [2]:
total_games = len(df_chess)
black_wins=df_chess[df_chess["Result"]=='0-1']
white_wins=df_chess[df_chess["Result"]=='1-0']

# Fraction or probablity
p_black_win=len(black_wins)/total_games
p_white_win=len(white_wins)/total_games

print(p_black_win)
print(p_white_win)

0.5
0.5


In [8]:
target_opening ="e4 c5 Nf3 Nc6 Bb5 g6"

# Likelihoods of both opeingings black and white
# Total unique opeings count

"Laplace Smoothing"
unique_opeings=len(df_chess['Opening'].unique())
alpha=1

# likelihood of balck
opening_in_black=len(black_wins[black_wins['Opening']==target_opening])

p_opening_given_black=(opening_in_black+alpha)/(unique_opeings+len(black_wins))

# White likelihoods

opening_in_white=len(white_wins[white_wins['Opening']==target_opening])

p_opening_given_white=(opening_in_white+alpha)/(unique_opeings+len(white_wins))


print(p_opening_given_black)
print(p_opening_given_white)



0.3333333333333333
0.16666666666666666


In [12]:
black_score=p_black_win*p_opening_given_black
white_score=p_white_win*p_opening_given_white


if(black_score>white_score):
  print(target_opening,"is black win")
else:
    print(target_opening," is white win")


0.16666666666666666   0.08333333333333333
e4 c5 Nf3 Nc6 Bb5 g6 is black win
